# Step 1 check 1 - Spatial, temporal and multivariate coherence of the climate imprint + QDM product

**Ad hoc test, outside the chain**: it reads the step-1 product of the supervisor's pipeline (EC-Earth3 climate
imprint + QDM at 0.25 deg, `qdm/merged`), ERA5 at 0.25 deg and the raw EC-Earth3 inputs (`guido_inputs`), and writes
only a cache on EOS (`outputs/step1_check/cache/`) and figures / tables in `outputs/` next to this notebook.

**Question.** Is the analogue block of the pipeline (block 3: regimes, analogues, reranking of the days within regime
episodes) needed? It is run only if the step-1 product shows a coherence deficit that it can correct.

**What the step-1 product inherits.** The QDM is monotonic at every pixel: it keeps the order of the days. The product
therefore has the order of the days, the persistence and the dependence between variables of EC-Earth3 at the GCM
scale, and below the GCM cell the spatial structure of the climate imprint: the GCM anomaly interpolated bilinearly
(tasmax, dtr) or the pattern of a best-correlated reanalysis day (pr). The analogues take the spatial patterns of
real reanalysis days and reorder the days within each regime episode (about 6 days): they can correct what lies below
the GCM scale (fine spatial structure, dependence between variables at that scale, order of the days within an
episode), not the errors of the GCM at the large scale (regime frequency and duration, persistence between episodes).

**Method.** Calibration period `YEARS` (1981-2014), where the marginal distributions of the product equal those of
ERA5 by construction, so that only the structure is tested. Five datasets:
- at 0.25 deg: `ERA5_025` (reference) and `PROD_025` (step-1 product);
- on the EC-Earth3 grid: `ERA5_07` and `PROD_07` (both aggregated: mean of the 0.25 deg pixels whose centre falls in
  the GCM cell) and `GCM_07` (raw EC-Earth3).

A deficit of the product at 0.25 deg that does not appear at 0.7 deg (`PROD_07` close to `ERA5_07`) is created by the
downscaling and could be corrected by the analogues; a deficit already present at 0.7 deg (`GCM_07` and `PROD_07`
far from `ERA5_07`) comes from the GCM.

Domain: LISFLOOD domain without the land outside Europe (`ba_utils.load_domain_mask`), `N_PIX` random pixels
(area-weighted) at 0.25 deg for the pixel statistics, all domain cells at 0.7 deg. Seasons DJF and JJA. Anomalies:
departure from the monthly climatology of each pixel (tasmax, dtr); pr as such.

| coherence | statistics |
|---|---|
| spatial | mean correlation of the daily series of all pairs of pixels per distance class, e-folding distance; standard deviation of the daily departure of each 0.25 deg pixel from the mean of its GCM cell (variability below the GCM cell); pr areal extremes: Rx1day, Rx5day of the mean over blocks of about 75, 225 and 450 km, areal reduction factor ARF = block Rx1day / mean Rx1day of its pixels |
| temporal | lag-1 autocorrelation; tasmax: summer hot spells (anomaly above its JJA 90th percentile: mean and mean annual maximum length); pr: P(wet / wet), P(wet / dry), mean wet and dry spells, CDD, CWD (wet = 1 mm/day) |
| multivariate | Spearman correlation of each pair of variables; tasmax and dtr anomalies on wet days minus dry days |

**Reading.** Table of section 4: for each statistic, the error of the product against ERA5 at 0.25 deg, the error of
the aggregated product and of the raw GCM against ERA5 at 0.7 deg, and the **downscaling gap** = error at 0.25 deg -
error of `PROD_07` at 0.7 deg. Flags with the tolerances `TOL_REL` / `TOL_ABS` (to agree with the supervisor before
reading the results; the values below are only placeholders). The variables whose product exists are used (pr is
added when its chain is done).

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, glob, time, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.stats import rankdata
import ba_utils as bu
import validation_utils as vu
warnings.filterwarnings("ignore")

CONFIG_PATH = None
cfg = bu.load_config(CONFIG_PATH)
RUN = os.environ.get("STEP1_RUN", "stab2050nohos")          # run folder of the pipeline (its 1981-2014 part is checked)
BASE = Path(cfg["data_root"]) / "elisa" / "GCM_downscaling"
RUN_DIRS = BASE / "downscaling"                               # run folders created by guido_pipeline/setup_run.sh
INP = BASE / "guido_inputs"                                   # inputs written by guido_pipeline/prepare_inputs.sh
ALL_VARS = ["tasmax", "dtr", "pr"]
YEARS = (1981, 2014)
N_PIX = 3000                                                  # random domain pixels at 0.25 deg (pixel statistics)
SEED = 20261120
SEASONS = {"DJF": (12, 1, 2), "JJA": (6, 7, 8)}
WET = vu.WET_DEP                                              # 1 mm/day
BINS = {"fine": np.arange(0, 1525, 25), "coarse": vu.DEP_BINS_KM}   # distance classes of the correlation (km)
BLOCKS = {"fine": (3, 9, 18), "coarse": (1, 3, 6)}           # k x k cells: about 75, 225, 450 km at both resolutions
MIN_COVER = 0.9                                               # GCM cells kept if covered at least this much by the 0.25 deg box
TOL_REL, TOL_ABS = 0.10, 0.05                                 # flags of section 4 (placeholders, to agree on)
OVERWRITE = False

CACHE = bu.out_dir(cfg, "step1_check", "cache")
OUT = Path.cwd() / "outputs"; OUT.mkdir(exist_ok=True)
TAG = f"{RUN}_{YEARS[0]}-{YEARS[1]}"
print(f"run {RUN} | years {YEARS[0]}-{YEARS[1]} | cache {CACHE}")

## 1. Files, grids, domain, aggregation onto the EC-Earth3 grid

In [ ]:
def product_file(var):
    run_dir = RUN_DIRS / f"climdown_cdo_{var}_europe_EC-Earth3-{RUN}_stab"
    fs = sorted(glob.glob(str(run_dir / "data" / "*" / "out" / "qdm" / "merged" / f"*01_12*{var}*.nc")))
    fs = [f for f in fs if "unique_qdm_" not in Path(f).name]      # monthly merges of postprocess_qdm, not the product
    if len(fs) != 1:
        print(f"  {var}: {len(fs)} product files in {run_dir}/data/*/out/qdm/merged: {[Path(f).name for f in fs]}")
        return None
    return Path(fs[0])


FILES = {v: {"ERA5": INP / "era5" / f"{v}_era5_1981_2014_europe.nc",
             "GCM": INP / RUN / f"{v}_EC-Earth3-{RUN}_HISTORICAL_europe.nc",
             "PROD": product_file(v)} for v in ALL_VARS}
VARS = [v for v in ALL_VARS if FILES[v]["PROD"] is not None and FILES[v]["ERA5"].exists() and FILES[v]["GCM"].exists()]
for v in ALL_VARS:
    print(f"{v:6s} " + " | ".join(f"{k}: {p.name if p is not None and p.exists() else 'MISSING'}" for k, p in FILES[v].items()))
print("variables checked:", VARS)
assert VARS, "no step-1 product yet"


def open_var(path, var):
    # daily field of YEARS, standard coordinates (ascending lat, lon -180..180), singleton dimensions dropped
    ds = xr.open_dataset(path, chunks={"time": 365}, **bu._TIME_KW)
    da = ds[var] if var in ds.data_vars else ds[list(ds.data_vars)[0]]
    for d in list(da.dims):
        if d not in ("time", "lat", "lon", "latitude", "longitude", "y", "x") and da.sizes[d] == 1:
            da = da.isel({d: 0}, drop=True)
    da = bu.norm_coords(da).transpose("time", "lat", "lon")
    t = da.indexes["time"]
    keep = (t.year >= YEARS[0]) & (t.year <= YEARS[1]) & ~((t.month == 2) & (t.day == 29))
    return da.isel(time=np.flatnonzero(keep))


F0 = open_var(FILES[VARS[0]]["ERA5"], VARS[0])
G0 = open_var(FILES[VARS[0]]["GCM"], VARS[0])
T_YEAR = np.asarray(F0.indexes["time"].year); T_MONTH = np.asarray(F0.indexes["time"].month)
NT = T_YEAR.size
assert NT == 365 * (YEARS[1] - YEARS[0] + 1), f"{NT} days in ERA5"
FLAT, FLON, GLAT, GLON = F0.lat.values, F0.lon.values, G0.lat.values, G0.lon.values
NF, NC = FLAT.size * FLON.size, GLAT.size * GLON.size

# 0.25 deg pixel -> EC-Earth3 cell containing its centre
ye, xe = bu._edges(GLAT), bu._edges(GLON)
iy = np.searchsorted(ye, FLAT) - 1; ix = np.searchsorted(xe, FLON) - 1
CELL = np.where(((iy >= 0) & (iy < GLAT.size))[:, None] & ((ix >= 0) & (ix < GLON.size))[None, :],
                iy[:, None] * GLON.size + ix[None, :], -1).ravel()
inside = np.flatnonzero(CELL >= 0)
CNT = np.bincount(CELL[inside], minlength=NC)
AGG = sparse.csr_matrix((1.0 / CNT[CELL[inside]], (CELL[inside], inside)), shape=(NC, NF))   # mean of the pixels of a cell
f_area = np.cos(np.deg2rad(FLAT))[:, None] * np.ones((1, FLON.size)) * 0.25 * 0.25
g_area = (np.cos(np.deg2rad(GLAT))[:, None] * np.diff(xe)[None, :] * np.diff(ye)[:, None])
COVER = (np.bincount(CELL[inside], weights=f_area.ravel()[inside], minlength=NC) / g_area.ravel()).reshape(GLAT.size, GLON.size)

DOM = {"fine": bu.load_domain_mask(cfg, "era5", like=F0.isel(time=0)).values,
       "coarse": bu.load_domain_mask(cfg, "ecearth3", like=G0.isel(time=0)).values & (COVER >= MIN_COVER)}
COORD = {"fine": (FLAT, FLON), "coarse": (GLAT, GLON)}
DOMPIX = {g: np.flatnonzero(DOM[g].ravel()) for g in DOM}
assert (CELL[DOMPIX["fine"]] >= 0).all(), "domain pixels outside the EC-Earth3 grid"

# pixels of the pixel statistics: N_PIX random domain pixels (area-weighted) at 0.25 deg, all domain cells at 0.7 deg
rng = np.random.default_rng(SEED)
w = np.cos(np.deg2rad(np.repeat(FLAT, FLON.size)))[DOMPIX["fine"]]
SAMPLE = {"fine": np.sort(rng.choice(DOMPIX["fine"], size=min(N_PIX, DOMPIX["fine"].size), replace=False, p=w / w.sum())),
          "coarse": DOMPIX["coarse"]}


def latlon(g, idx):
    la, lo = COORD[g]
    return la[idx // lo.size], lo[idx % lo.size]


def block_matrix(g, k, min_cells=0.5):
    # domain pixels -> blocks of k x k cells with at least min_cells of them in the domain (mean of the block)
    lo = COORD[g][1].size
    p = DOMPIX[g]; bid = (p // lo // k) * 100000 + (p % lo) // k
    ub, inv, cnt = np.unique(bid, return_inverse=True, return_counts=True)
    keep = np.flatnonzero(cnt >= min_cells * k * k)
    m = np.isin(inv, keep)
    col = np.searchsorted(keep, inv[m])
    return sparse.csr_matrix((1.0 / cnt[inv[m]], (np.flatnonzero(m), col)), shape=(p.size, keep.size))


BLK = {g: {k: block_matrix(g, k) for k in BLOCKS[g]} for g in BLOCKS}
for g in ("fine", "coarse"):
    print(f"{g:6s}: {DOM[g].sum()} domain pixels, {SAMPLE[g].size} in the pixel statistics, blocks "
          + ", ".join(f"{k}x{k}: {BLK[g][k].shape[1]}" for k in BLOCKS[g]))

## 2. Extraction (one year at a time, cached on EOS)

For each dataset and variable: the daily series of the sample pixels, the daily means of the blocks, the yearly
maxima of the domain pixels (for the ARF of pr), and at 0.25 deg the sums of the daily departure from the mean of the
GCM cell (per season and pixel).

In [ ]:
DATASETS = {"ERA5_025": ("ERA5", "fine"), "PROD_025": ("PROD", "fine"),
            "ERA5_07": ("ERA5", "coarse"), "PROD_07": ("PROD", "coarse"), "GCM_07": ("GCM", "coarse")}
SEAS_OF_MONTH = {m: s for s, ms in SEASONS.items() for m in ms}


def to_coarse(flat):
    # (days, NF) 0.25 deg fields -> (days, NC) means of the pixels of each EC-Earth3 cell
    return np.asarray((AGG @ flat.T).T)


def extract(name, var):
    path = CACHE / f"{name}_{var}_{TAG}.npz"
    if path.exists() and not OVERWRITE:
        return dict(np.load(path))
    src, g = DATASETS[name]
    t0 = time.time()
    da = open_var(FILES[var][src], var)
    assert da.sizes["time"] == NT, f"{name} {var}: {da.sizes['time']} days"
    if src == "GCM":
        assert bu.same_grid(da, G0)
    else:
        assert bu.same_grid(da, F0), f"{name} {var}: not on the ERA5 0.25 deg grid"
    S, B, PX, SUB = [], {k: [] for k in BLOCKS[g]}, [], np.zeros((3, len(SEASONS), DOMPIX["fine"].size))
    yrs = np.asarray(da.indexes["time"].year)
    for y in range(YEARS[0], YEARS[1] + 1):
        ti = np.flatnonzero(yrs == y)
        flat = da.isel(time=ti).values.reshape(ti.size, -1).astype("float64")
        if g == "fine":
            d = flat[:, DOMPIX["fine"]] - to_coarse(flat)[:, CELL[DOMPIX["fine"]]]   # departure from the GCM-cell mean
            mon = T_MONTH[T_YEAR == y]
            for si, s in enumerate(SEASONS):
                ds_ = d[np.isin(mon, SEASONS[s])]
                SUB[0, si] += ds_.shape[0]; SUB[1, si] += ds_.sum(0); SUB[2, si] += (ds_ ** 2).sum(0)
        if src != "GCM" and g == "coarse":
            flat = to_coarse(flat)
        S.append(flat[:, SAMPLE[g]].astype("float32"))
        dom = flat[:, DOMPIX[g]]
        for k in BLOCKS[g]:
            B[k].append(np.asarray((BLK[g][k].T @ dom.T).T, dtype="float32"))
        PX.append(dom.max(0).astype("float32"))
    out = {"S": np.concatenate(S), "PX": np.stack(PX), **{f"B{k}": np.concatenate(B[k]) for k in BLOCKS[g]}}
    if g == "fine":
        n, s1, s2 = SUB[0], SUB[1], SUB[2]
        out["SUBSD"] = np.sqrt(np.maximum(s2 / n - (s1 / n) ** 2, 0)).astype("float32")   # (season, domain pixel)
    np.savez(path, **out)
    print(f"  {name} {var}: {time.time() - t0:.0f} s")
    return out


E = {}
for var in VARS:
    for name in DATASETS:
        E[name, var] = extract(name, var)
    print(f"{var}: extracted")

## 3. Statistics

In [ ]:
def series(name, var):
    x = E[name, var]["S"].astype("float64")
    return x if var == "pr" else vu._anomalies(x, T_MONTH)


def wmean(v, g, idx):
    la, _ = latlon(g, idx)
    w = np.cos(np.deg2rad(la)); ok = np.isfinite(v)
    return float((w[ok] * v[ok]).sum() / w[ok].sum())


def lag1(x, months):
    sel = np.isin(T_MONTH, months)
    i = np.flatnonzero(sel[:-1] & sel[1:])
    return vu._pearson_cols(x[i], x[i + 1])


def hot_spells(a):
    # JJA days with the anomaly above its JJA 90th percentile: mean spell length and mean annual maximum (per pixel)
    jja = np.isin(T_MONTH, SEASONS["JJA"])
    thr = np.quantile(a[jja], 0.9, axis=0)
    rows, yr = [], []
    for y in range(YEARS[0], YEARS[1] + 1):
        s = jja & (T_YEAR == y)
        rows += [a[s] > thr, np.zeros((1, a.shape[1]), bool)]       # a dry row between summers: no spell across years
        yr += [np.full(s.sum() + 1, y)]
    sp = vu._spells(np.concatenate(rows), np.concatenate(yr))
    return sp[0], sp[3]


ROWS = []


def add(var, aspect, stat, season, kind, values):
    ROWS.append(dict(var=var, aspect=aspect, statistic=stat, season=season, kind=kind, **values))


GRID = {n: g for n, (_, g) in DATASETS.items()}
CORR = {}
for var in VARS:
    X = {n: series(n, var) for n in DATASETS}
    for s, months in SEASONS.items():
        add(var, "temporal", "lag-1 autocorrelation", s, "abs",
            {n: wmean(lag1(X[n], months), GRID[n], SAMPLE[GRID[n]]) for n in DATASETS})
        sel = np.isin(T_MONTH, months)
        for n in DATASETS:
            la, lo = latlon(GRID[n], SAMPLE[GRID[n]])
            c, r, _ = vu.spatial_correlation(X[n][sel], la, lo, bins=BINS[GRID[n]])
            CORR[var, s, n] = (c, r)
        add(var, "spatial", "e-folding distance (km)", s, "rel",
            {n: vu.efold_km(*CORR[var, s, n]) for n in DATASETS})
        add(var, "spatial", "correlation at the first distance class", s, "abs",
            {n: CORR[var, s, n][1][np.isfinite(CORR[var, s, n][1])][0] for n in DATASETS})
        si = list(SEASONS).index(s)
        add(var, "spatial", "SD of the departure from the GCM-cell mean (0.25 deg)", s, "rel",
            {n: (wmean(E[n, var]["SUBSD"][si], "fine", DOMPIX["fine"]) if "SUBSD" in E[n, var] else np.nan) for n in DATASETS})
    if var == "tasmax":
        hs = {n: hot_spells(X[n]) for n in DATASETS}
        add(var, "temporal", "hot spell, mean length (days)", "JJA", "rel", {n: wmean(hs[n][0], GRID[n], SAMPLE[GRID[n]]) for n in DATASETS})
        add(var, "temporal", "hot spell, annual max length (days)", "JJA", "rel", {n: wmean(hs[n][1], GRID[n], SAMPLE[GRID[n]]) for n in DATASETS})
    if var == "pr":
        wet = {n: X[n] >= WET for n in DATASETS}
        for s, months in SEASONS.items():
            sel = np.isin(T_MONTH, months)
            i = np.flatnonzero(sel[:-1] & sel[1:])
            with np.errstate(invalid="ignore", divide="ignore"):
                add(var, "temporal", "P(wet | wet)", s, "abs",
                    {n: wmean((wet[n][i] & wet[n][i + 1]).sum(0) / wet[n][i].sum(0), GRID[n], SAMPLE[GRID[n]]) for n in DATASETS})
                add(var, "temporal", "P(wet | dry)", s, "abs",
                    {n: wmean((~wet[n][i] & wet[n][i + 1]).sum(0) / (~wet[n][i]).sum(0), GRID[n], SAMPLE[GRID[n]]) for n in DATASETS})
        sp = {n: vu._spells(wet[n], T_YEAR) for n in DATASETS}
        for k, lab in enumerate(("mean wet spell (days)", "mean dry spell (days)", "CDD (days)", "CWD (days)")):
            add(var, "temporal", lab, "year", "rel", {n: wmean(sp[n][k], GRID[n], SAMPLE[GRID[n]]) for n in DATASETS})
        # areal extremes: blocks of about 75, 225, 450 km (same block rank at both resolutions)
        for bi, km in enumerate((75, 225, 450)):
            vals = {"Rx1day": {}, "Rx5day": {}, "ARF": {}}
            for n in DATASETS:
                g = GRID[n]; k = BLOCKS[g][bi]; b = E[n, var][f"B{k}"].astype("float64")
                yr = T_YEAR
                rx1 = np.stack([b[yr == y].max(0) for y in range(YEARS[0], YEARS[1] + 1)]).mean(0)
                cs = np.vstack([np.zeros((1, b.shape[1])), np.cumsum(b, 0)]); b5 = cs[5:] - cs[:-5]
                rx5 = np.stack([b5[yr[:b5.shape[0]] == y].max(0) for y in range(YEARS[0], YEARS[1] + 1)]).mean(0)
                px = E[n, var]["PX"].mean(0)                                 # mean annual max of each domain pixel
                M = (BLK[g][k] > 0).astype(float)
                arf = rx1 / ((M.T @ px) / np.asarray(M.sum(0)).ravel())
                vals["Rx1day"][n], vals["Rx5day"][n], vals["ARF"][n] = np.median(rx1), np.median(rx5), np.median(arf)
            for st, v in vals.items():
                add(var, "spatial", f"areal {st}, blocks ~{km} km (median)", "year", "rel", v)

# multivariate: the same sample pixels for every variable
for a_, b_ in [(a_, b_) for i, a_ in enumerate(VARS) for b_ in VARS[i + 1:]]:
    Xa = {n: series(n, a_) for n in DATASETS}; Xb = {n: series(n, b_) for n in DATASETS}
    for s, months in SEASONS.items():
        sel = np.isin(T_MONTH, months)
        add(f"{a_}-{b_}", "multivariate", f"Spearman {a_} - {b_}", s, "abs",
            {n: wmean(vu._pearson_cols(rankdata(Xa[n][sel], axis=0), rankdata(Xb[n][sel], axis=0)), GRID[n], SAMPLE[GRID[n]])
             for n in DATASETS})
if "pr" in VARS:
    P = {n: E[n, "pr"]["S"] >= WET for n in DATASETS}
    for v in [v for v in VARS if v != "pr"]:
        X = {n: series(n, v) for n in DATASETS}
        for s, months in SEASONS.items():
            sel = np.isin(T_MONTH, months)
            def wd(n):
                x, w_ = X[n][sel], P[n][sel]
                with np.errstate(invalid="ignore"):
                    return np.nanmean(np.where(w_, x, np.nan), 0) - np.nanmean(np.where(~w_, x, np.nan), 0)
            add(f"{v}-pr", "multivariate", f"{v} anomaly, wet minus dry days (degC)", s, "abs",
                {n: wmean(wd(n), GRID[n], SAMPLE[GRID[n]]) for n in DATASETS})
print(len(ROWS), "statistics")

## 4. Errors against ERA5 at both scales, and the downscaling gap

`err_025` = PROD_025 against ERA5_025, `err_07_prod` = PROD_07 against ERA5_07, `err_07_gcm` = GCM_07 against
ERA5_07 (relative error for lengths, distances, extremes and standard deviations; difference for correlations and
probabilities); `gap` = err_025 - err_07_prod: the part of the error created below the GCM cell, which the analogues
could correct. `flag`: |gap| above the tolerance. At 0.7 deg the ~75 km block is a single cell (ARF = 1 by definition).

In [ ]:
RES = pd.DataFrame(ROWS)


def err(a, b, kind):
    return (a / b - 1.0) if kind == "rel" else (a - b)


RES["err_025"] = [err(r.PROD_025, r.ERA5_025, r.kind) for r in RES.itertuples()]
RES["err_07_prod"] = [err(r.PROD_07, r.ERA5_07, r.kind) for r in RES.itertuples()]
RES["err_07_gcm"] = [err(r.GCM_07, r.ERA5_07, r.kind) for r in RES.itertuples()]
RES["gap"] = RES["err_025"] - RES["err_07_prod"]
RES["flag"] = [("DEFICIT BELOW THE GCM CELL" if abs(r.gap) > (TOL_REL if r.kind == "rel" else TOL_ABS) else "")
               for r in RES.itertuples()]
RES.loc[RES["statistic"].str.contains("GCM-cell"), ["err_07_prod", "err_07_gcm", "gap"]] = np.nan
RES.loc[RES["statistic"].str.contains("GCM-cell"), "flag"] = [
    ("DEFICIT BELOW THE GCM CELL" if abs(e) > TOL_REL else "") for e in RES.loc[RES["statistic"].str.contains("GCM-cell"), "err_025"]]
RES.to_csv(OUT / f"coherence_{TAG}.csv", index=False)
cols = ["var", "aspect", "statistic", "season", "ERA5_025", "PROD_025", "ERA5_07", "PROD_07", "GCM_07",
        "err_025", "err_07_prod", "err_07_gcm", "gap", "flag"]
with pd.option_context("display.width", 260, "display.max_columns", None, "display.max_colwidth", 55,
                       "display.float_format", "{:.3f}".format, "display.max_rows", 200):
    display(RES[cols])
print("saved:", OUT / f"coherence_{TAG}.csv")

## 5. Figures: correlation against distance, variability below the GCM cell, areal extremes

In [ ]:
STYLE = {"ERA5_025": ("k", "-"), "PROD_025": ("C3", "-"), "ERA5_07": ("k", "--"), "PROD_07": ("C3", "--"), "GCM_07": ("C0", "--")}
fig, axes = plt.subplots(len(VARS), len(SEASONS), figsize=(6 * len(SEASONS), 3.6 * len(VARS)), squeeze=False)
for i, var in enumerate(VARS):
    for j, s in enumerate(SEASONS):
        ax = axes[i, j]
        for n in DATASETS:
            c, r = CORR[var, s, n]
            ax.plot(c, r, color=STYLE[n][0], ls=STYLE[n][1], label=n)
        ax.axhline(np.exp(-1), c="0.6", lw=0.6); ax.set_xlim(0, 1000); ax.set_ylim(0, 1)
        ax.set_title(f"{var}{' anomaly' if var != 'pr' else ''}, {s}", fontsize=9)
        ax.set_xlabel("distance (km)"); ax.set_ylabel("mean correlation")
axes[0, 0].legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT / f"correlation_distance_{TAG}.png", dpi=110); plt.show()

BORDERS = bu.borders(cfg, "10M")
fig, axes = plt.subplots(len(VARS), len(SEASONS), figsize=(6.5 * len(SEASONS), 4 * len(VARS)), squeeze=False)
for i, var in enumerate(VARS):
    for j, s in enumerate(SEASONS):
        ratio = np.full(NF, np.nan)
        ratio[DOMPIX["fine"]] = E["PROD_025", var]["SUBSD"][j] / E["ERA5_025", var]["SUBSD"][j]
        ax = axes[i, j]
        pm = ax.pcolormesh(FLON, FLAT, ratio.reshape(FLAT.size, FLON.size), shading="nearest", cmap="RdBu", vmin=0, vmax=2)
        bu.draw_borders(ax, BORDERS, lw=0.3)
        ax.set_title(f"{var} {s}: SD of the departure from the GCM-cell mean, product / ERA5", fontsize=9)
        plt.colorbar(pm, ax=ax, shrink=0.8)
plt.tight_layout(); plt.savefig(OUT / f"subgrid_variability_{TAG}.png", dpi=110); plt.show()

if "pr" in VARS:
    sub = RES[RES["statistic"].str.startswith("areal")]
    fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
    for ax, st in zip(axes, ("Rx1day", "Rx5day", "ARF")):
        for n in DATASETS:
            d = sub[sub["statistic"].str.contains(f"areal {st},")]
            ax.plot([75, 225, 450], d[n].values, marker="o", color=STYLE[n][0], ls=STYLE[n][1], label=n)
        ax.set_xlabel("block size (km)"); ax.set_title(f"pr areal {st} (median over blocks)", fontsize=9)
    axes[0].legend(fontsize=8)
    plt.tight_layout(); plt.savefig(OUT / f"areal_extremes_{TAG}.png", dpi=110); plt.show()

## 6. Summary

Statistics flagged (gap or, for the variability below the GCM cell, error at 0.25 deg above the tolerance): the
deficits that the downscaling creates below the GCM cell and that the analogue block could correct. The decision is
taken with the supervisor on these numbers.

In [ ]:
flag = RES[RES["flag"] != ""][["var", "aspect", "statistic", "season", "err_025", "err_07_prod", "err_07_gcm", "gap"]]
print(f"{len(flag)} of {len(RES)} statistics flagged (TOL_REL {TOL_REL}, TOL_ABS {TOL_ABS}); variables: {VARS}")
with pd.option_context("display.width", 220, "display.max_colwidth", 60, "display.float_format", "{:.3f}".format):
    display(flag)